# Run on Colab, from VS Code or Cursor

1. Install Google's **Colab** extension, open this notebook locally, then *Select Kernel → Colab → New Colab Server → L4*.
2. Run all. Code runs on the Colab machine: the repo is cloned to `/content/latent-dispatch` there, and results land in `/content/latent-dispatch/results/`, not on Drive.
3. Get the results back: turn on the extension's server mounting (setting *Colab: Server Mounting*), then copy `results/` from the Colab server's folder into your local repo's `results/`. The last cell also writes one tarball.

**Keep the session alive overnight.** The Colab machine's disk is wiped when its runtime ends, and Colab Pro doesn't keep running after the client disconnects (background execution is a Pro+ feature). Keep the laptop awake (`caffeinate -dis` in a terminal) and VS Code open. Every question is saved as soon as it finishes, so rerunning after a kernel restart (same runtime) picks up where it stopped.

Stages, each of which stops the notebook if it fails: machine check → code → install → benchmarks → tests (including this model's real-weights check) → smoke run of every notebook + audit → full run → pack.

In [1]:
# What to run. One model per run: check its results before moving to a bigger model.
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SEEDS = [0]
N_PER_CELL = 100                                  # quick preliminary; None = each notebook's full design
NOTEBOOKS = ["part0_ruler", "h1_encoding", "h2_thinking"]
SKIP = {"h2_thinking": ["t1_1024"]}               # the expensive T1 control, left out of the quick pass
RUN_TESTS = True
MIN_BOXED, MAX_CUT = 90.0, 1.0                    # smoke gate, percent of answers per arm

REPO = "https://github.com/JoshBong/latent-dispatch.git"
BRANCH = "main"
ROOT = "/content/latent-dispatch"

In [2]:
import os
import subprocess
import sys

import torch


def sh(cmd):
    """Run a shell command and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


print(sys.version)
assert torch.cuda.is_available(), "no GPU: connect to a Colab GPU server (L4)"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
if DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise SystemExit("this GPU has no bf16 (a T4?): pick an L4 or A100")

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NVIDIA L4 22 GB


In [3]:
# The code. Either upload latent-dispatch.bundle (right-click it in the Explorer -> "Upload to Colab"; it lands in
# /content) and it is cloned from that, no token needed; or it is cloned from GitHub with a token that can read the
# repo (used once, never saved or printed). To switch to a newer commit, delete /content/latent-dispatch and rerun.
import getpass

BUNDLE, BUNDLE_BRANCH = "/content/latent-dispatch.bundle", "colab-run"
if not os.path.exists(ROOT) and os.path.exists(BUNDLE):
    sh(f"git clone -q -b {BUNDLE_BRANCH} {BUNDLE} {ROOT}")
elif not os.path.exists(ROOT):
    token = getpass.getpass("GitHub token with read access to the repo: ").strip()
    url = REPO.replace("https://", f"https://x-access-token:{token}@")      # any username works with a token
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, ROOT], capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})      # fail instead of waiting for a password
    message = (r.stderr or "").replace(token, "***")
    del token, url
    if r.returncode != 0:
        raise RuntimeError("git clone failed:\n" + message)
    subprocess.run(["git", "-C", ROOT, "remote", "set-url", "origin", REPO], check=True)
os.chdir(ROOT)
sh("git log -1 --format='%h %s'")

RuntimeError: git clone failed: check the token and that it can read the repo

In [ ]:
sh("pip install -q -e '.[dev]'")       # pins transformers 4.57.6; torch comes with Colab
sys.path.insert(0, ROOT)

In [ ]:
names = ["toy", "toy_d0", "musique", "hotpotqa"]
missing = [n for n in names if not os.path.exists(f"benchmark/{n}.jsonl")]
if missing:
    sh("python -m latent_dispatch.data " + " ".join(missing))
sh("cd benchmark && sha256sum -c SHA256SUMS")

In [ ]:
if RUN_TESTS:
    sh("python -m pytest -q")                                                   # fast suite, ~1 min
    sh(f"MODEL={MODEL} DTYPE={DTYPE} DEVICE=cuda python -m pytest -q -m slow tests/test_real_model.py")

In [ ]:
# Smoke: every notebook on 2 questions per cell, then the audit. Stops here if any arm's answers aren't usable.
import logging

import papermill as pm

from latent_dispatch import analysis, results

log = logging.getLogger("papermill")
log.setLevel(logging.INFO)
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))
os.makedirs("notebooks/runs", exist_ok=True)
tag = MODEL.split("/")[-1]


def run_notebook(name, smoke):
    out = f"notebooks/runs/{tag}-{name}{'-smoke' if smoke else ''}.ipynb"
    pm.execute_notebook(f"notebooks/{name}.ipynb", out, kernel_name="python3", log_output=True,
                        parameters=dict(MODEL=MODEL, DTYPE=DTYPE, SMOKE=smoke, SEEDS=SEEDS, N_PER_CELL=N_PER_CELL,
                                        SKIP=SKIP.get(name, [])))


for name in NOTEBOOKS:
    run_notebook(name, smoke=True)
table = analysis.audit(results.load_all(MODEL, DTYPE))
display(table)
bad = table[(table["boxed %"] < MIN_BOXED) | (table["cut %"] > MAX_CUT)]
if len(bad):
    raise SystemExit(f"smoke audit failed for {list(bad.index)}: read the raw outputs in notebooks/runs/*-smoke.ipynb "
                     "before the full run")

In [ ]:
# Full run. Resumes: questions already saved (including the smoke's) are skipped.
for name in NOTEBOOKS:
    run_notebook(name, smoke=False)
display(analysis.audit(results.load_all(MODEL, DTYPE)))

In [ ]:
sh(f"tar czf /content/results-{tag}-{DTYPE}.tar.gz results notebooks/runs")
print(f"copy {ROOT}/results/ (or /content/results-{tag}-{DTYPE}.tar.gz) into your local repo")